In [ ]:
# KAGGLE ML BENCHMARK + FASTAPI — COMPLETE ONE-CELL VERSION
# Model inputs: .pt and .joblib ONLY. Dataset inputs: .csv ONLY.
# In Kaggle, attach your files using Add Input, then run this cell.
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, torch
import ipywidgets as w
from IPython.display import display, clear_output
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUT.mkdir(parents=True, exist_ok=True)
MODELS = {}

def scan():
    paths = sorted((p for p in ROOT.rglob('*') if p.is_file()), key=str)
    # Exclude known preprocessing objects. Do not interpret JSON manifests as models.
    models = [p for p in paths if p.suffix.lower() in {'.pt', '.joblib'}
              and not any(s in p.stem.lower() for s in ('preprocessor', 'scaler', 'encoder', 'imputer'))]
    # Show every CSV separately; no filename-based deduplication.
    csvs = [p for p in paths if p.suffix.lower() == '.csv']
    return models, csvs

model_ui = w.SelectMultiple(description='Models', rows=7, layout=w.Layout(width='98%'))
csv_ui = w.Dropdown(description='Dataset', layout=w.Layout(width='98%'))
target_ui = w.Dropdown(description='Target', layout=w.Layout(width='75%'))
task_ui = w.Dropdown(description='Task', options=['classification', 'regression'])
refresh_btn = w.Button(description='Refresh inputs')
load_btn = w.Button(description='Load models', button_style='success')
check_btn = w.Button(description='Check inputs')
bench_btn = w.Button(description='Benchmark', button_style='primary')
query_btn = w.Button(description='Query FastAPI')
log = w.Output()

def set_targets(change=None):
    previous = target_ui.value
    if not csv_ui.value:
        target_ui.options = []
        return
    try:
        cols = pd.read_csv(csv_ui.value, nrows=1).columns.tolist()
        target_ui.options = cols
        if cols: target_ui.value = previous if previous in cols else cols[-1]
    except Exception as e:
        target_ui.options = []
        with log: print('Cannot read selected CSV:', e)

def refresh(_=None):
    old_models = set(model_ui.value)
    old_csv = csv_ui.value
    m, c = scan()
    model_ui.options = [(str(p), str(p)) for p in m]
    model_ui.value = tuple(x for x in old_models if x in {str(p) for p in m})
    csv_ui.options = [(str(p), str(p)) for p in c]
    if old_csv in {str(p) for p in c}: csv_ui.value = old_csv
    set_targets()
    with log:
        clear_output(wait=True)
        print(f'Found {len(m)} model files (.pt/.joblib) and {len(c)} CSV files.')
        if not c: print('No CSV found under', ROOT, '- attach a CSV via Kaggle Add Input.')
        if not m: print('No model found - attach your model via Kaggle Add Input.')

# A .pt may be TorchScript OR a state_dict checkpoint. Never assume one format.
def load_one(path):
    if Path(path).suffix.lower() == '.joblib':
        obj = joblib.load(path)  # Use trusted artifacts only.
        if not callable(getattr(obj, 'predict', None)):
            raise ValueError(f'{type(obj).__name__} is not a prediction model')
        return obj, 'estimator'
    try:
        obj = torch.jit.load(path, map_location='cpu')
        obj.eval()
        return obj, 'torchscript'
    except Exception:
        ckpt = torch.load(path, map_location='cpu', weights_only=True)
        if isinstance(ckpt, dict) and 'state_dict' in ckpt:
            raise ValueError('PyTorch checkpoint (state_dict), not TorchScript; requires original nn.Module architecture. '
                             f"dimensions={ckpt.get('dimensions')}, task={ckpt.get('task')}")
        raise ValueError(f'Unsupported .pt content: {type(ckpt).__name__}')

def load(_=None):
    MODELS.clear()
    with log:
        clear_output(wait=True)
        for path in model_ui.value:
            try:
                obj, kind = load_one(path)
                name = Path(path).stem
                i = 2
                while name in MODELS:
                    name = f'{Path(path).stem}_{i}'; i += 1
                MODELS[name] = {'obj':obj, 'kind':kind, 'path':path}
                print('LOADED:', name, '|', kind)
            except Exception as e:
                print('NOT LOADED:', path, '\n ', e)
        print('Models available for inference:', len(MODELS))

def data():
    if not csv_ui.value: raise ValueError('No CSV selected')
    df = pd.read_csv(csv_ui.value)
    if target_ui.value not in df.columns: raise ValueError('Choose a target column')
    df = df.dropna(subset=[target_ui.value])
    if df.empty: raise ValueError('CSV has no labeled records')
    return df.drop(columns=[target_ui.value]), df[target_ui.value]

def predict(name, X):
    item = MODELS[name]; obj = item['obj']
    if item['kind'] == 'estimator':
        if hasattr(obj, 'feature_names_in_'):
            expected = list(obj.feature_names_in_)
            missing = [c for c in expected if c not in X.columns]
            if missing: raise ValueError(f'Missing features: {missing}')
            X = X.loc[:, expected]
        return np.asarray(obj.predict(X)).reshape(-1)
    with torch.no_grad():
        raw = obj(torch.tensor(X.to_numpy(dtype=np.float32)))
        if isinstance(raw, (list, tuple)): raw = raw[0]
        raw = raw.detach().cpu().numpy()
    if task_ui.value == 'regression': return raw.reshape(-1)
    if raw.ndim == 2 and raw.shape[1] > 1: return raw.argmax(axis=1)
    return (raw.reshape(-1) >= 0.5).astype(int)  # Assumes binary probabilities.

def check(_=None):
    with log:
        clear_output(wait=True)
        try:
            X, _ = data()
            print('Dataset:', csv_ui.value, '|', len(X), 'rows')
            print('Target:', target_ui.value, '| Features:', list(X.columns))
            if not MODELS: print('No trained prediction model loaded yet.')
            for name in MODELS:
                try: print('PASS:', name, predict(name, X.iloc[:1]).tolist())
                except Exception as e: print('FAIL:', name, str(e))
        except Exception as e: print('INPUT ERROR:', e)

def benchmark():
    if not MODELS: raise ValueError('Load a prediction model first')
    X, y = data(); rows = []
    for name, item in MODELS.items():
        try:
            predict(name, X.iloc[:min(2, len(X))])
            times = []
            for _ in range(5):
                start = time.perf_counter()
                pred = predict(name, X)
                times.append((time.perf_counter()-start)*1000)
            row = {'model':name, 'kind':item['kind'], 'rows':len(X),
                   'median_ms':round(float(np.median(times)), 3)}
            if task_ui.value == 'classification':
                row.update(accuracy=accuracy_score(y, pred),
                           f1=f1_score(y, pred, average='weighted', zero_division=0))
            else:
                row.update(rmse=float(np.sqrt(mean_squared_error(y, pred))),
                           r2=r2_score(y, pred))
            rows.append(row)
        except Exception as e:
            rows.append({'model':name, 'error':str(e)})
    board = pd.DataFrame(rows)
    metric = 'f1' if task_ui.value == 'classification' else 'rmse'
    if metric in board:
        board = board.sort_values(metric, ascending=(metric == 'rmse'), na_position='last')
    board.to_csv(OUT/'benchmark_leaderboard.csv', index=False)
    return board

app = FastAPI(title='Kaggle Multi-Model Serving Lab')
class Request(BaseModel):
    instances: list[dict]

@app.get('/health')
def health(): return {'ok':True, 'loaded':len(MODELS)}

@app.get('/models')
def models(): return [{'name':n, 'kind':v['kind']} for n,v in MODELS.items()]

@app.post('/predict/{name}')
def predict_one(name: str, body: Request):
    if name not in MODELS: raise HTTPException(status_code=404, detail='Unknown model')
    try: return {'model':name, 'predictions':predict(name, pd.DataFrame(body.instances)).tolist()}
    except Exception as e: raise HTTPException(status_code=422, detail=str(e))

@app.post('/predict-all')
def predict_all(body: Request):
    X = pd.DataFrame(body.instances); results = {}
    for name in MODELS:
        try: results[name] = predict(name, X).tolist()
        except Exception as e: results[name] = {'error':str(e)}
    return {'results':results}

client = TestClient(app)

def bench_click(_=None):
    with log:
        clear_output(wait=True)
        try:
            display(benchmark())
            print('Saved:', OUT/'benchmark_leaderboard.csv')
        except Exception as e: print('BENCHMARK ERROR:', e)

def query(_=None):
    with log:
        clear_output(wait=True)
        try:
            X, _ = data()
            payload = {'instances':json.loads(X.iloc[:1].to_json(orient='records'))}
            print('GET /health:', client.get('/health').json())
            print('GET /models:', client.get('/models').json())
            response = client.post('/predict-all', json=payload)
            print('POST /predict-all HTTP', response.status_code)
            print(json.dumps(response.json(), indent=2))
        except Exception as e: print('QUERY ERROR:', e)

csv_ui.observe(set_targets, names='value')
refresh_btn.on_click(refresh)
load_btn.on_click(load)
check_btn.on_click(check)
bench_btn.on_click(bench_click)
query_btn.on_click(query)

display(w.HTML('<h3>Kaggle ML Benchmark + FastAPI Lab</h3>'))
display(model_ui, csv_ui, target_ui, task_ui)
display(w.HBox([refresh_btn, load_btn, check_btn, bench_btn, query_btn]), log)
refresh()
print('Ready: choose models and one CSV, load, check, benchmark, query.')
# END OF SINGLE CELL
